<a href="https://colab.research.google.com/github/abhiniveshg0-max/DEEP-LEARNING/blob/main/Experiment%204%20part%20a%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip -d intel_dataset

TRAIN_DIR = 'intel_dataset/seg_train/seg_train'
TEST_DIR = 'intel_dataset/seg_test/seg_test'

import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_full_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)

val_batches = tf.data.experimental.cardinality(val_full_ds)
val_ds = val_full_ds.take(val_batches // 2)
extra_test_ds = val_full_ds.skip(val_batches // 2)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

normalization_layer = tf.keras.layers.Rescaling(1. / 255)

def normalize(ds):
    return ds.map(lambda x, y: (normalization_layer(x), y))

train_ds = normalize(train_ds)
val_ds = normalize(val_ds)
test_ds = normalize(test_ds)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("\nNumber of batches:")
print("Training batches   :", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches :", tf.data.experimental.cardinality(val_ds).numpy())
print("Testing batches     :", tf.data.experimental.cardinality(test_ds).numpy())

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.tight_layout()
plt.show()


























